# Detector de hojas de café: fases 1 a 5 en Colab o Kaggle

Ejecuta las celdas **en orden**. Cada fase termina con una comprobación de sus criterios de aceptación; si falla, el cuaderno se detiene y no debes continuar.

Antes de empezar: `Entorno de ejecución > Cambiar tipo de entorno > GPU`.

Necesitas el archivo `cafe_ml_bundle.zip`, que se genera en el repositorio con `make colab-bundle`.

## 0. Código y dependencias

In [ ]:
import os, zipfile
if not os.path.exists('cafe_ml_bundle.zip'):
    from google.colab import files  # in Kaggle, add the zip as a dataset or upload it to the working directory instead
    files.upload()
zipfile.ZipFile('cafe_ml_bundle.zip').extractall('.')
%cd ml
!pip install -q -r requirements.txt 'tensorflow[and-cuda]==2.21.0'

In [ ]:
# TensorFlow needs the pip-installed CUDA libraries on the loader path, so every script runs through run().
import glob, json, os, site, subprocess, sys
nvidia = [p for s in site.getsitepackages() for p in glob.glob(os.path.join(s, 'nvidia', '*', 'lib'))]
ENV = dict(os.environ, LD_LIBRARY_PATH=':'.join(nvidia + [os.environ.get('LD_LIBRARY_PATH', '')]))

def run(*args):
    subprocess.run([sys.executable, *args], env=ENV, check=True)

def load(name):
    return json.load(open(f'outputs/{name}'))

run('-c', "import tensorflow as tf; g = tf.config.list_physical_devices('GPU'); print('TensorFlow', tf.__version__, 'GPU:', g); assert g, 'No GPU: enable it in the runtime settings'")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Fase 1. Datos

Descarga RoCoLe (Mendeley Data) y DECAFIA (Zenodo), ambos CC BY 4.0, y construye las particiones.

In [ ]:
!mkdir -p data/raw/rocole data/raw/decafia
!curl -sSfL --retry 5 -C - -o data/raw/rocole/RoCoLE-json.json https://data.mendeley.com/public-files/datasets/c5yvn32dzg/files/654cdf31-4ce6-4a15-b5cf-52a92181a698/file_downloaded
!curl -sSfL --retry 5 -C - -o data/raw/rocole/RoCoLe-voc.tar.gz https://data.mendeley.com/public-files/datasets/c5yvn32dzg/files/3cfa348c-179b-4f05-9a6d-6c9343dd5c03/file_downloaded
!curl -sSfL --retry 5 -C - -o data/raw/decafia/CoffeeLeaf-CO.zip 'https://zenodo.org/records/19931904/files/CoffeeLeaf-CO.zip?download=1'
assert os.path.getsize('data/raw/rocole/RoCoLe-voc.tar.gz') == 697591506, 'RoCoLe download incomplete: run this cell again'
assert os.path.getsize('data/raw/decafia/CoffeeLeaf-CO.zip') == 208325327, 'DECAFIA download incomplete: run this cell again'
!tar -xzf data/raw/rocole/RoCoLe-voc.tar.gz -C data/raw/rocole
!unzip -q -o data/raw/decafia/CoffeeLeaf-CO.zip -d data/raw/decafia

In [ ]:
run('01_preparar_datos.py')
stats = load('dataset_stats.json')
assert all(v == 0 for v in stats['leakage_check'].values()), stats['leakage_check']
assert min(stats['counts']['train'].values()) >= 100, stats['counts']['train']
print('ACEPTACIÓN FASE 1: OK | imágenes', stats['final_images'], '| train', stats['counts']['train'], '| desbalance', stats['imbalance_ratio_train'])

## Fase 2. Entrenamiento

In [ ]:
run('02_entrenar.py')
history = load('history.json')
for name in ('model.keras', 'labels.txt', 'history.json'):
    assert os.path.exists(f'outputs/{name}'), name
val_loss = history['head']['val_loss'] + history['finetune']['val_loss']
assert min(val_loss) < val_loss[0], 'validation loss never improved over epoch 1'
print('ACEPTACIÓN FASE 2: OK | val_loss época 1', round(val_loss[0], 4), '| mínima', round(min(val_loss), 4))
print('accuracy final train/val (cabeza):', round(history['head']['accuracy'][-1], 4), round(history['head']['val_accuracy'][-1], 4))
print('accuracy final train/val (ajuste fino):', round(history['finetune']['accuracy'][-1], 4), round(history['finetune']['val_accuracy'][-1], 4))
print(history['training'])

Revisa las curvas: si la exactitud de entrenamiento queda muy por encima de la de validación hay sobreajuste y hay que reportarlo.

## Fase 3. Evaluación y umbral

In [ ]:
run('03_evaluar.py')
for name in ('metrics.json', 'confusion_matrix.png', 'threshold.json'):
    assert os.path.exists(f'outputs/{name}'), name
print('ACEPTACIÓN FASE 3: OK')
from IPython.display import Image, display
display(Image('outputs/confusion_matrix.png'))

## Fase 4. Conversión a TFLite

In [ ]:
run('04_convertir_tflite.py')
assert load('model_io.json')['contract_ok']
print('ACEPTACIÓN FASE 4: OK')

## Fase 5. Paridad y vectores de referencia

In [ ]:
run('05_paridad_tflite.py')
parity = load('parity.json')
assert os.path.exists('outputs/golden/golden.json')
agreement = parity['keras_vs_quantized']['argmax_agreement']
print('ACEPTACIÓN FASE 5: archivos generados | coincidencia de clase Keras vs cuantizado:', agreement, '(esperado >= 0.98)')
if agreement < 0.98:
    print('AVISO: por debajo de lo esperado; hay que reportarlo.')

## Descargar los resultados

Entrega `cafe_outputs.zip` completo: de ahí salen todas las cifras del informe.

In [ ]:
!zip -q -r ../cafe_outputs.zip outputs
try:
    from google.colab import files
    files.download('../cafe_outputs.zip')
except ImportError:
    print('Download cafe_outputs.zip from the working directory')